# ScholarPulse UK — pipeline audit and live evaluation

CS5079 Applied AI, University of Aberdeen. The cells below read the exported classifier and the 113,233-project Gateway to Research snapshot collected on 23 November 2026. Nothing here is a synthetic label or a stand-in probability.

The live service does not retrain. `artifacts/tfidf_lr_weights.npz` is the portable export of the scikit-learn pipeline (200,000-feature sublinear TF-IDF, class-weighted logistic regression, temperature T = 1.1314305006097425).

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "projects_meta.parquet").exists():
    ROOT = Path.cwd().parent
frame = pd.read_parquet(ROOT / "data" / "projects_meta.parquet")
info = json.loads((ROOT / "artifacts" / "model_info.json").read_text())
print("projects", len(frame))
print("columns", list(frame.columns))
banned = {"pi_name", "researcher", "person_name", "investigator", "orcid", "surname"}
print("personal columns", banned.intersection({c.lower() for c in frame.columns}) or "none")
print("councils", frame["lead_funder"].value_counts().to_dict())
print("snapshot field", info.get("data_collected"), "n_index", info.get("n_projects_index"))
print("macro-F1", info["test_macro_f1"], "accuracy", info["test_accuracy"], "top-3", info["test_top3"])
print("temperature", info["temperature"])
print("calibration")
print(pd.DataFrame(info["calibration"]).to_string(index=False))
assert len(frame) == 113233
assert info["test_macro_f1"] == 0.8615
assert abs(info["temperature"] - 1.1314305006097425) < 1e-12


projects 113233
columns ['project_id', 'grant_ref', 'title', 'lead_funder', 'grant_category', 'lead_org_name', 'department', 'award_gbp', 'fund_start', 'fund_end', 'gtr_url', 'start_year']
personal columns none
councils {'EPSRC': 31270, 'Innovate UK': 24544, 'BBSRC': 12809, 'MRC': 10465, 'AHRC': 10306, 'ESRC': 9554, 'NERC': 8774, 'STFC': 5511}
snapshot field 2026-11-23T19:35:16+00:00 n_index 113233
macro-F1 0.8615 accuracy 0.8693 top-3 0.9916
temperature 1.1314305006097425
calibration
                                     probabilities    ECE  Brier  log loss  accuracy
                           raw (class-weighted LR) 0.0336 0.1954    0.3846    0.8693
temperature-scaled (T = 1.13, fitted on OOF train) 0.0180 0.1935    0.3755    0.8693


## Live inference

One EPSRC-shaped demo and the interdisciplinary borderline demo. Probabilities come from `softmax(logits / T)` on the exported weights. The contrast row is `(w_EPSRC − w_InnovateUK) · x`.

In [2]:
import time
from fundmatch.demos import DEMOS
from fundmatch.engine import get_engine

engine = get_engine()
print(f"loaded {len(engine.frame)} projects in {engine.cold_s:.2f}s, RSS {engine.rss_mb():.0f} MB")
quantum = next(item for item in DEMOS if item["id"] == "epsrc-quantum")
border = next(item for item in DEMOS if item["id"] == "borderline")
t0 = time.perf_counter()
result = engine.analyze(quantum["title"], quantum["abstract"], top_k=5)
print(f"warm query {result.latency_ms:.1f} ms (wall {(time.perf_counter()-t0)*1000:.1f} ms)")
print("top", result.top_council, result.councils[0].probability, result.confidence)
print("probabilities", [(row.council, row.probability) for row in result.councils])
print("supporting", [(t.feature, t.contribution) for t in result.explanation.positive[:6]])
print("award median", result.award.median, "P25", result.award.p25, "P75", result.award.p75)
print("zero excluded", result.award.n_zero_excluded, "of", result.award.neighbours)
print("similar", [(row.lead_funder, row.title[:70]) for row in result.similar])
contrast = engine.contrast(border["title"], border["abstract"], "EPSRC", "Innovate UK")
print("toward EPSRC", [t.feature for t in contrast.toward_a])
print("toward Innovate UK", [t.feature for t in contrast.toward_b])
real = engine.analyze(quantum["title"], quantum["abstract"], cpih=True)
print("nominal median", result.award.median, "real median", real.award.median)
assert result.ok and abs(sum(row.probability for row in result.councils) - 1) < 1e-5
assert real.award.median != result.award.median


loaded 113233 projects in 2.76s, RSS 662 MB
warm query 104.2 ms (wall 104.4 ms)
top EPSRC 0.892068 high
probabilities [('EPSRC', 0.892068), ('Innovate UK', 0.098302), ('STFC', 0.006696), ('NERC', 0.001821), ('MRC', 0.000626), ('BBSRC', 0.000266), ('ESRC', 0.000134), ('AHRC', 8.8e-05)]
supporting [('quantum', 0.884338), ('superconducting', 0.540958), ('compiler', 0.465549), ('reinforcement learning', 0.390717), ('gate', 0.366462), ('reinforcement', 0.346404)]
award median 191858.5 P25 93642.0 P75 579254.5
zero excluded 22 of 30
similar [('EPSRC', 'Scalable two-dimensional silicon quantum processor'), ('EPSRC', 'Two-qubit interactions in two spatial dimensions in silicon'), ('EPSRC', 'Mediating two-qubit interactions over long distances in silicon-based '), ('EPSRC', 'Intrinsically protected superconducting quantum circuits'), ('Innovate UK', 'Compilation & Circuit Layout Optimisation For Superconducting Quantum ')]
toward EPSRC ['research', 'proposal', 'engineering', 'industrial partner

## Int8 weight fidelity

Lecture 8 quantises the coefficient matrix per class. Top-1 agreement is measured on the five demos plus 40 real project titles. The float64 model remains the one that answers queries.

In [3]:
from fundmatch.quantize import export_int8, fidelity

report = export_int8(engine)
texts = [f"{item['title']}. {item['abstract']}" for item in DEMOS]
texts += engine.frame["title"].head(40).tolist()
fit = fidelity(engine, texts, report.pop("coef_hat"))
print("int8 file MB", round(report["int8_file_bytes"] / 1e6, 3))
print("float32 coef MB", round(report["fp32_coef_bytes"] / 1e6, 3))
print(fit)
assert fit["top1_agreement"] >= 0.9


int8 file MB 0.776
float32 coef MB 6.4
{'n': 50, 'top1_agreement': 0.98, 'max_probability_delta': 0.030091, 'macro_f1_vs_fp64': 0.990023, 'ms_per_text': 0.146}


## v2 checks

Conformal sets follow the published test profile. Novelty, the 2D map, a what-if edit, and the int8 weights are scored on the same exported model.

In [4]:
from fundmatch.demos import DEMOS
quantum = next(item for item in DEMOS if item["id"] == "epsrc-quantum")
border = next(item for item in DEMOS if item["id"] == "borderline")
peaked = engine.analyze(quantum["title"], quantum["abstract"])
print("90% set", peaked.conformal_set_90)
print("95% set", peaked.conformal_set_95)
print("novelty", peaked.novelty["band"], peaked.novelty["max_sim"], peaked.novelty["percentile"])
print("constellation kinds", sorted({p["kind"] for p in peaked.constellation}))
print("hosts", peaked.host_institutions[:3])
print("schemes", peaked.scheme_fit[:3])
edit = engine.simulate_what_if(border["title"], border["abstract"], add_terms="clinical cohort cytokine", target_council="MRC")
print("MRC before/after", edit["target"])
fast = engine.analyze(quantum["title"], quantum["abstract"])
small = engine.analyze(quantum["title"], quantum["abstract"], quantized=True)
print(f"fp64 {fast.top_council} {fast.councils[0].probability:.3f} in {fast.latency_ms:.1f} ms")
print(f"int8 {small.top_council} {small.councils[0].probability:.3f} in {small.latency_ms:.1f} ms")
assert peaked.conformal_set_90 == ["EPSRC"]
assert small.precision == "int8"


90% set ['EPSRC']
95% set ['EPSRC', 'Innovate UK', 'STFC']
novelty Core UKRI Territory 0.8864 85.6
constellation kinds ['centroid', 'landmark', 'neighbour', 'query']
hosts [{'organisation': 'UNIVERSITY COLLEGE LONDON', 'projects': 4, 'median_gbp': None, 'top_council': 'EPSRC'}, {'organisation': 'CARDIFF UNIVERSITY', 'projects': 1, 'median_gbp': None, 'top_council': 'EPSRC'}, {'organisation': 'IMPERIAL COLLEGE LONDON', 'projects': 1, 'median_gbp': None, 'top_council': 'EPSRC'}]
schemes [{'scheme': 'Studentship', 'projects': 9, 'share': 0.9}, {'scheme': 'Collaborative R&D', 'projects': 1, 'share': 0.1}]
MRC before/after {'council': 'MRC', 'before': 0.014992, 'after': 0.03603, 'delta': 0.021037}


fp64 EPSRC 0.892 in 92.3 ms
int8 EPSRC 0.888 in 91.4 ms


## v3 student tools

Working titles cite real projects. The wave is a count ratio inside the 200 nearest dated grants. Readiness is five documented rules, not a mark.

In [5]:
topics = engine.suggest_topics("school exclusion earnings English towns education records", "undergrad")
print("topics", len(topics["topics"]))
print(topics["topics"][0]["working_title"])
print("precedent", topics["topics"][0]["precedents"][0]["project_id"], topics["topics"][0]["precedents"][0]["organisation"])
wave = engine.topic_trends("wheat drought gene family water table")
print("wave", wave["band"], wave["note"])
score = engine.readiness(DEMOS[0]["title"], DEMOS[0]["abstract"], "masters")
print("readiness", score["overall"], [d["id"] for d in score["dimensions"]])
bridge = engine.fuse("music", "machine learning")
print("bridge councils", bridge["bridge_councils"])
print("bridge project", bridge["projects"][0]["title"], bridge["projects"][0]["organisation"])
assert len(topics["topics"]) == 5
assert wave["band"] in {"Surging", "Rising", "Steady", "Mature", "insufficient"}
assert 0 <= score["overall"] <= 100


topics 5
Refugee: a bounded study of educational
precedent 8F47C815-2AB9-4616-B06E-C7056E24CEF2 UNIVERSITY OF NOTTINGHAM
wave Mature 16 projects in 2024–2026 versus 28 earlier (ratio 0.57). The neighbourhood is quieter than it was.
readiness 69.9 ['specificity', 'alignment', 'originality', 'method', 'bridge']


bridge councils ['EPSRC', 'Innovate UK']
bridge project Fine-grained music source separation with deep learning models AUDIOSTRIP LTD
